# Initialization

In [1]:
# read the data
import matplotlib.pyplot as plt
import numpy as np
#import pandas as pd
import matplotlib

import mne
import os
from mne.preprocessing import ICA
%matplotlib qt

In [2]:
# understand the 
import sys
sys.path.append(os.path.abspath('..'))
from utils import identify_bad_channels,remove_epochs_with_bad_mmn_channels, Average_in_trials_in_time_window, remove_epochs_with_n_bad_channels ,remove_or_interpolate_epochs_with_n_bad_chn
import importlib
import utils
importlib.reload(utils)

#from utils import remove_or_interpolate_epochs


<module 'utils' from 'e:\\ecf-exp2-notif-mmn\\analysis\\utils.py'>

In [3]:
from mne.utils import set_config
from mne.utils import get_config
set_config('MNE_USE_CUDA', 'true')
mne.set_log_level('WARNING')  # Options: 'DEBUG', 'INFO', 'WARNING', 'ERROR', 'CRITICAL'
# Check that the config is set
print(get_config('MNE_USE_CUDA')) 
#mne.set_memmap_min_size('100M')

true


# Data Reading 

In [4]:
data_folder = r"D:\Work_data\Notification_evoked_response\Data_at_IITD/bids_data/"
subjects = [sub for sub in os.listdir(data_folder) if (("sub-" in sub) & (os.path.isdir(os.path.join(data_folder,sub))))]
print(subjects)
data_type = "eeg"
 # it is useful to scroll eeg data 
B1_MMN_erp_evoked_dict  = {}
B1_devi_eps_evoked_dict  = {} 
B1_sta_eps_evoked_dict  = {}                            
B2_devi_eps_evoked_dict = {}
B2_sta_eps_evoked_dict = {}
B2_MMN_erp_evoked_dict = {}
epoch_per_num = {}
epoch_per_num["subject Dropped"] = []


['sub-sd011', 'sub-sd012', 'sub-sd013', 'sub-sd014', 'sub-sd015', 'sub-sd016', 'sub-sd017', 'sub-sd018', 'sub-sd019', 'sub-sd020', 'sub-sd021', 'sub-sd022', 'sub-sd023', 'sub-sd024', 'sub-sd025', 'sub-sd026', 'sub-sd027', 'sub-sd028', 'sub-sd029', 'sub-sd030', 'sub-sd031', 'sub-sd032', 'sub-sd033', 'sub-sd034', 'sub-sd035', 'sub-sd036', 'sub-sd037', 'sub-sd038', 'sub-sd039', 'sub-sd040', 'sub-sd041', 'sub-sd042', 'sub-sd043', 'sub-sd044', 'sub-sd045', 'sub-sd046', 'sub-sd047', 'sub-sd048', 'sub-sd049', 'sub-sd050', 'sub-sd051', 'sub-sd052', 'sub-sd053', 'sub-sd054', 'sub-sd055', 'sub-sd056', 'sub-sd057', 'sub-sd058', 'sub-sd059', 'sub-sd060', 'sub-sd061', 'sub-sd062', 'sub-sd063', 'sub-sd064', 'sub-sd065', 'sub-sd066', 'sub-sd067']


# Visual inspections and marking bad channels 

In [5]:
sub = 'sub-sd029'
path = os.path.join(data_folder,sub,data_type)
    
for eeg in os.listdir(path):
        if eeg[-5:] == ".vhdr":
           
            raw = mne.io.read_raw_brainvision(os.path.join(path,eeg), preload=True)
            #raw = mne.add_reference_channels(raw, ref_channels=["Cz"])
            raw.set_montage("easycap-M1")
            raw.plot(scalings=dict(eeg=40e-6) )

In [6]:
raw.info

Measurement date,Unknown
Experimenter,Unknown
Participant,Unknown
Digitized points,67 points
Good channels,61 EEG
Bad channels,"F3, CP2, FC2"
EOG channels,Not available
ECG channels,Not available
Sampling frequency,1000.00 Hz
Highpass,0.00 Hz
Lowpass,500.00 Hz


# Pre processing

In [7]:
raw.set_montage("easycap-M1")
#raw.interpolate_bads(reset_bads=True)
raw.set_eeg_reference(ref_channels=['TP9', 'TP10'])
raw.resample(512)
            # filter 
raw.filter(l_freq=0.1,h_freq=30  )
raw.plot(scalings=dict(eeg=40e-6) )

# ICA 

In [8]:
#ICA  b 
ica = ICA(n_components=None, random_state=97)
ica.fit(raw)
print("ICA Completed ...")
ica.plot_components()

ICA Completed ...


[<MNEFigure size 975x967 with 20 Axes>,
 <MNEFigure size 975x967 with 20 Axes>,
 <MNEFigure size 975x967 with 20 Axes>,
 <MNEFigure size 195x260 with 1 Axes>]

In [ ]:
eog_channels = ['Fp1', 'Fp2',"AF7","AF8",'F7', 'F8']  # Adjust these to match your channel names
eog_indices, eog_scores = ica.find_bads_eog(raw, ch_name=eog_channels)

print(eog_indices, eog_scores)

In [9]:
ica.exclude

[0, 2]

In [10]:
ica.exclude  = [0, 2,]
ica.apply(raw)
raw.plot(scalings=dict(eeg=50e-6))

# Interpolate bad

In [11]:
raw.interpolate_bads(reset_bads=True)

Measurement date,Unknown
Experimenter,Unknown
Participant,Unknown
Digitized points,67 points
Good channels,"64 EEG, 2 misc"
Bad channels,None
EOG channels,Not available
ECG channels,Not available
Sampling frequency,512.00 Hz
Highpass,0.10 Hz
Lowpass,30.00 Hz


# Epoching

In [12]:
events,event_dict = mne.events_from_annotations(raw)
print(events,event_dict)


[[      0       0   10013]
 [      0       0   10012]
 [      0       0   10013]
 ...
 [1813945       0   10006]
 [1814961       0   10004]
 [1814961       0   10004]] {'Stimulus/S   11': 10001, 'Stimulus/S   14': 10002, 'Stimulus/S   15': 10003, 'Stimulus/S   21': 10004, 'Stimulus/S   24': 10005, 'Stimulus/S   25': 10006, 'Stimulus/S   31': 10007, 'Stimulus/S   32': 10008, 'Stimulus/S   33': 10009, 'Stimulus/S   34': 10010, 'Stimulus/S   35': 10011, 'Stimulus/S10001': 10012, 'Stimulus/S99999': 10013}


In [13]:
# Create Epochs without droping the eye blink event
event_id = {
                'S14': event_dict['Stimulus/S   14'],
                'S15': event_dict['Stimulus/S   15'],
                'S24': event_dict['Stimulus/S   24'],
                'S25': event_dict['Stimulus/S   25'],
                'S34': event_dict['Stimulus/S   34'],
                'S35': event_dict['Stimulus/S   35'],}
epochs = mne.Epochs(raw, events, event_id=event_id,baseline=(None, 0) , tmin=-0.300, tmax=0.900, preload=True,event_repeated='merge')
epochs.drop_channels(['GSR', 'PPG'])

Number of events,2700
Events,S14: 750S15: 150S24: 750S25: 150S34: 750S35: 150
Time range,-0.301 – 0.900 s
Baseline,-0.301 – 0.000 s


In [14]:
epochs.plot(events=events,
                     scalings=dict(eeg=20e-6),
                     n_epochs=50,)


C:\Users\PRAKASH\AppData\Local\Temp\ipykernel_9512\4032285812.py:1: RuntimeWarning: More events than default colors available. You should pass a list of unique colors.
  epochs.plot(events=events,


In [ ]:
# Interpolated and 

# Drop or Interpolate Bad epochs  

In [15]:
cleaned_epochs = remove_or_interpolate_epochs_with_n_bad_chn(epochs,  bad_chn_threshold = 6, amplitude_threshold=80e-6)

e:\ecf-exp2-notif-mmn\analysis\utils.py:188: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data = epochs.get_data()


['Fp1', 'Fz', 'F3', 'Fp2', 'AF7', 'AF3', 'AFz', 'F1', 'AF4'] [0, 1, 2, 30, 31, 32, 33, 34, 60]
[] []
['Fp1'] [0]
['AFz'] [33]
['FT7'] [36]
[] []
[] []
[] []
[] []
['Fp1', 'Fz', 'F3', 'FT9', 'F1', 'CPz'] [0, 1, 2, 4, 34, 51]
['Fp1', 'Fp2', 'AF3', 'AFz', 'AF4', 'FCz'] [0, 30, 32, 33, 60, 62]
['Fp1', 'Fz', 'F3', 'FC1', 'FC2', 'AF3', 'AFz', 'F1', 'FC3', 'C1', 'CPz', 'F2', 'FCz'] [0, 1, 2, 6, 27, 32, 33, 34, 37, 38, 51, 61, 62]
['AFz'] [33]
['AFz'] [33]
['AFz'] [33]
['CPz'] [51]
['F6'] [58]
['F6', 'F2'] [58, 61]
[] []
['F2'] [61]
['F2'] [61]
['F2'] [61]
['F6', 'F2'] [58, 61]
[] []
[] []
[] []
[] []
[] []
['F2'] [61]
['F2'] [61]
['F2'] [61]
['CPz', 'F2'] [51, 61]
[] []
[] []
['F2'] [61]
['F2'] [61]
['F2'] [61]
['F2'] [61]
['Fp1', 'Fz', 'F3', 'FC1', 'FC2', 'AFz', 'F1', 'F5', 'FC3', 'F2', 'FCz'] [0, 1, 2, 6, 27, 33, 34, 35, 37, 61, 62]
['Fp1', 'F3', 'AF3', 'F1', 'F2'] [0, 2, 32, 34, 61]
['FC1', 'C1', 'CPz', 'C2', 'F2', 'FCz', 'Cz'] [6, 38, 51, 55, 61, 62, 63]
['F2'] [61]
['F2'] [61]
['F2'] [61

e:\ecf-exp2-notif-mmn\analysis\utils.py:215: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  return mne.concatenate_epochs(good_epochs)
e:\ecf-exp2-notif-mmn\analysis\utils.py:215: RuntimeWarning: Event number greater than 2147483647 created, events[:, 0] will be assigned consecutive increasing integer values
  return mne.concatenate_epochs(good_epochs)


In [18]:
print(f"Original number of epochs: {len(epochs)}")
print(f"Number of epochs after removal: {len(cleaned_epochs)}")
cleaned_epochs.plot()

Original number of epochs: 2700
Number of epochs after removal: 2440


# Drop the mannual marked bad Epochs 

In [22]:
cleaned_epochs.drop_bad()
print(f"Original number of epochs: {len(epochs)}")
print(f"Number of epochs after removal: {len(cleaned_epochs)}")
cleaned_epochs.plot()

Original number of epochs: 2700
Number of epochs after removal: 2415


In [35]:
PN_devi_eps = cleaned_epochs['S15'].average()
BN_sta_eps = cleaned_epochs['S14'].average()

BN_devi_eps = cleaned_epochs['S25'].average() 
PN_sta_eps = cleaned_epochs['S24'].average() 
CN_devi_eps = cleaned_epochs['S35'].average() 
CN_sta_eps = cleaned_epochs['S34'].average() 


In [ ]:
evokeds = { 
    "SN MMN" : mne.combine_evoked([ PN_devi_eps,PN_sta_eps], weights=[1,-1]),
    'BN MMN':mne.combine_evoked([ BN_devi_eps,BN_sta_eps], weights=[1,-1]),
    "CN MMN" : mne.combine_evoked([ CN_devi_eps,CN_sta_eps], weights=[1,-1]),
           }

stylesdict ={"SN MMN": {"linewidth": 3,"linestyle":'solid'},
             "BN MMN" : {"linewidth": 3,"linestyle":'solid'},
             "CN MMN" : {"linewidth": 3,"linestyle":'solid'},
             }
mne.viz.plot_compare_evokeds(evokeds,styles = stylesdict,
                             truncate_xaxis=False,truncate_yaxis=False,picks=['Cz'],
                             ylim = dict(eeg=[-6.2,6.2]))

[<Figure size 800x600 with 1 Axes>]

# Save the Epochs data 

In [30]:
folder = r"E:\ecf-exp2-notif-mmn\data\Processed"
cleaned_epochs.save(f'{os.path.join(folder,sub)}-epo.fif', overwrite=True)

C:\Users\PRAKASH\AppData\Local\Temp\ipykernel_9512\2588044517.py:2: RuntimeWarning: epochs.drop_log contains 13244320 entries which will incur up to a 214.5 MB writing overhead (per split file), consider using epochs.reset_drop_log_selection() prior to writing
  cleaned_epochs.save(f'{os.path.join(folder,sub)}-epo.fif', overwrite=True)


# Read data and verify the results 

In [42]:
re_epochs = mne.read_epochs(f'{os.path.join(folder,sub)}-epo.fif', preload=True)

print(f"Number of epochs after re_epochs read: {len(re_epochs)}")

Number of epochs after re_epochs read: 2415


In [43]:
re_PN_devi_eps = re_epochs['S15'].average()
re_BN_sta_eps = re_epochs['S14'].average()

re_BN_devi_eps = re_epochs['S25'].average() 
re_PN_sta_eps = re_epochs['S24'].average() 
re_CN_devi_eps = re_epochs['S35'].average() 
re_CN_sta_eps = re_epochs['S34'].average() 

In [44]:
re_evokeds = { 
    "SN MMN" : mne.combine_evoked([ re_PN_devi_eps,re_PN_sta_eps], weights=[1,-1]),
    'BN MMN':mne.combine_evoked([ re_BN_devi_eps,re_BN_sta_eps], weights=[1,-1]),
    "CN MMN" : mne.combine_evoked([ re_CN_devi_eps,re_CN_sta_eps], weights=[1,-1]),
           }

stylesdict ={"SN MMN": {"linewidth": 3,"linestyle":'solid'},
             "BN MMN" : {"linewidth": 3,"linestyle":'solid'},
             "CN MMN" : {"linewidth": 3,"linestyle":'solid'},
             }
mne.viz.plot_compare_evokeds(re_evokeds,styles = stylesdict,
                             truncate_xaxis=False,truncate_yaxis=False,picks=['Fz'],
                             ylim = dict(eeg=[-6.2,6.2]))

[<Figure size 800x600 with 2 Axes>]